# FASE 5 - Preparación y diagnostico 


Este script reúne en una sola tabla todo lo que usan las partes posteriores de esta fase,  y deja un diagnóstico para tomar sus decisiones. 

Para cada uno de los eventos: 

1. Trae los resultados de la Fase 3 (distancia robusta, residuales y perfil) y las variables del modelo.

2. Marca si es anómalo, con los mismos cortes de la Fase 3.

3. Agrega su región de SKATER de la Fase 4.

4. Busca su unidad geológica, geomorfológica y de cobertura sin agrupar, que no entró a la GWR.

5. Mide la orientación de su eje mayor, la distancia a la falla más cercana y el ángulo entre el eje y esa falla.

**Nota**:  divide las fallas en tramos de 200 m con su orientación, para comparar la orientación general de eventos y fallas.


**Salidas**

En una nueva carpeta y gdb, correspondiente a la fase, entrega:

* tres capas: eventos (como puntos) con la información agregada, el regtángulo de área mínima de cada polígono, del que sale la orientación del eje mayor. Los tramos de 200 m de las fallas, recortados al área de estudio, con su orientación.

* tres archivos: Todos como tablas de entrada de las fases posteriores y de resultado de este script.



#  Librerías

Carga las librerías y activa la extensión Spatial Analyst.


In [36]:
import os   #arma las rutas
import math  # cálculo de ángulos 
 
import arcpy  # lectura y creación de capas 

# Manejo de tablas
import numpy as np   
import pandas as pd

# reemplazar las capas de una corrida anterior.
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

'CheckedOut'

# Configuración 

### **Qué hace**

Reúne las rutas, los nombres de capas y campos, y los parámetros de la fase. Es lo único que hay que editar.

In [37]:

#rutas 
# Carpetnas, gdb del proyecto, gdb de salida y carpeta con los csv

BASE = r"D:\Users\tasal\Desktop\Tesis de grado"
GDB_IN = r"C:\Users\villa\Documents\ArcGIS\Projects\GDB_proyecto de grado\GDB_proyecto de grado.gdb"   # GDB con fallas, lineamientos, ventana y polígonos/rectángulos
GDB_OUT = r"D:\Users\tasal\Desktop\Tesis de grado\fase5\Fase5.gdb"
CSV_DIR = r"D:\Users\tasal\Desktop\Tesis de grado\fase5\Fase5_csv"


# Resultados por evento y variables del modelo.

CSV_EVENTOS_R = os.path.join(BASE, "fase3", "F3_fase3_integrado.csv")
TABLA_MODELO = os.path.join(BASE, "fase3", "F3_S2_tabla_modelo.csv")

# Tabla con la región de cada evento y la columna que se toma.

TABLA_REGIONES = r"D:\Users\tasal\Desktop\Tesis de grado\fase4\F4_R_clustgeo_k5.csv"      
F_REG = "SKATER"                                                          


# Capas 

RECTANG = os.path.join(GDB_IN, "fase1_movilidad")
POLIGONOS = os.path.join(GDB_IN, "Inventario_integrado")       # solo si RECTANG no existe
RECT_METODO = "RECTANGLE_BY_AREA"                             # tipo de rectangulo
AREA_ESTUDIO = os.path.join(GDB_IN, "area_VA")                   
FALLAS = os.path.join(GDB_IN, "fallas_lineamientos")             # fallas de la microzonificación


# Para cada campo nuevo (GEO_UNI, GEOM_UNI, COB_UNI), 
#la capa de polígonos y el campo con el nombre de la unidad.


UNIDADES = {
    "GEO_UNI": (os.path.join(GDB_IN, "Geologia"), "NOMBRE"),        
    "GEOM_UNI": (os.path.join(GDB_IN, "Geomorfologia"), "NOMBRE"), 
    "COB_UNI": (os.path.join(GDB_IN, "cobertura_amva"), "NOMBRE"),       
}

# Campo agrupado (mismo que se usó en la GWR) que corresponde a cada campo sin agrupar

AGRUPADO = {"GEO_UNI": "GEOLOGIA", "GEOM_UNI": "GEOMORFOLOGIA", "COB_UNI": "COBERTURA"}


# Columnas de los CSV
F_ID, F_X, F_Y = "ID", "X", "Y"
F_DR = "RD"                        # distancia robusta SIN elevar al cuadrado
F_ID_POL = "ID_evento"             # ID en la capa de rectángulos/polígonos
 
# Parámetros
UMB_MOD, UMB_FUE = 9.35, 11.34     # chi² 0,975 y 0,99 con 3 gl (sobre RD²)
RADIO_BUFFER_M = 18                # mismo círculo de la Fase 3
TRAMO_M = 200                      # tramos de falla y ventana de la orientación local
BANDA_M, DMAX_M = 500, 5000
E_MIN = 1.2                        # elongación mínima para orientación definida
DIST_ALINEACION_M = 1000           # sensibilidad de la alineación
MIN_EV_UNIDAD = (15, 20)           # Cortes de eventos por unidad

# Funciones

### ¿Qué hace?

Define las funciones de lectura, de geometría y de cruce con las capas.

* `nombres_campos`, `agregar_campo` y `verificar_capa`: listan los campos de una capa, agregan un campo si no existe y comprueban que una capa exista, tenga el tipo de geometría esperado y el campo pedido.

* `leer_csv` y `exportar_csv` leen un CSV limpiando los nombres de columna, y exportan una capa a CSV sin geometría.
 
* `orientacion(p1, p2)`  el ángulo de un segmento medido desde el norte de la cuadrícula, entre 0 y 180°.

* `eje_mayor(geom)` toma el lado más largo del rectángulo; su orientación es la del eje mayor y la elongación es el lado largo entre el corto.

* `leer_rectangulos()`  ajusta a cada polígono su rectángulo de área mínima, lo guarda y  devuelve la orientación y la elongación de cada evento.

* `unidad_mayoritaria(buffers, capa, campo)` cruza los círculos de 18 m con una capa de unidades, suma el área de cada unidad dentro de cada círculo y se queda con la de mayor área. Devuelve también su pureza: la fracción del área cruzada que ocupa esa unidad.

In [38]:

def nombres_campos(fc):
    return [f.name for f in arcpy.ListFields(fc)]
 
 
def agregar_campo(fc, nombre, tipo):
    if nombre not in nombres_campos(fc):
        if tipo == "TEXT":
            arcpy.management.AddField(fc, nombre, tipo, field_length=120)
        else:
            arcpy.management.AddField(fc, nombre, tipo)
 
 
def verificar_capa(ruta, campo=None, tipo_geom=None):
    if not arcpy.Exists(ruta):
        raise FileNotFoundError(f"No existe la capa: {ruta}")
    d = arcpy.Describe(ruta)
    if tipo_geom and getattr(d, "shapeType", None) != tipo_geom:
        raise ValueError(f"{ruta} debe ser de tipo {tipo_geom} (es {getattr(d, 'shapeType', '?')})")
    if campo and campo not in nombres_campos(ruta):
        raise ValueError(f"El campo '{campo}' no está en {os.path.basename(ruta)}. "
                         f"Campos disponibles: {nombres_campos(ruta)}")
 
 
def leer_csv(ruta):
    t = pd.read_csv(ruta, encoding="utf-8-sig")
    t.columns = [str(c).lstrip("\ufeff").strip().strip('"') for c in t.columns]
    return t
 
 
def exportar_csv(fc, ruta):
    campos = [f.name for f in arcpy.ListFields(fc) if f.type not in ("Geometry", "OID")]
    t = pd.DataFrame([r for r in arcpy.da.SearchCursor(fc, campos)], columns=campos)
    t.to_csv(ruta, index=False, encoding="utf-8-sig")
    return t
 
 
def orientacion(p1, p2):
    return math.degrees(math.atan2(p2.X - p1.X, p2.Y - p1.Y)) % 180.0
 
 
def eje_mayor(geom):
    """Orientación axial y elongación de un rectángulo."""
    pts = [p for p in geom.getPart(0) if p]
    lados = []
    for a, b in zip(pts[:-1], pts[1:]):
        largo = math.hypot(b.X - a.X, b.Y - a.Y)
        if largo > 0:
            lados.append((largo, a, b))
    if len(lados) < 2:
        return None, None
    lados.sort(key=lambda x: x[0], reverse=True)
    return orientacion(lados[0][1], lados[0][2]), lados[0][0] / lados[-1][0]
 
 
def clave_id(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return str(v).strip()
 
 
def campo_id(fc):
    for c in (F_ID_POL, F_ID, "ID_EVENTO", "ID"):
        if c in nombres_campos(fc):
            return c
    raise ValueError(f"No se encontró un campo de ID en {os.path.basename(fc)}. "
                     f"Pon el nombre correcto en F_ID_POL. Campos: {nombres_campos(fc)}")
 
 
def leer_rectangulos():
    verificar_capa(POLIGONOS, tipo_geom="Polygon")
    idc = campo_id(POLIGONOS)
    oid = arcpy.Describe(POLIGONOS).OIDFieldName
    mapa = {o: i for o, i in arcpy.da.SearchCursor(POLIGONOS, [oid, idc])}
    rect_fc = os.path.join(GDB_OUT, "F5_rectangulos")
    arcpy.management.MinimumBoundingGeometry(POLIGONOS, rect_fc, RECT_METODO, "NONE")
    print(f"Rectángulos ajustados a {os.path.basename(POLIGONOS)} (campo de ID: {idc})")
    res = {}
    with arcpy.da.SearchCursor(rect_fc, ["ORIG_FID", "SHAPE@"]) as cur:
        for k, g in cur:
            if g is not None and k in mapa:
                res[clave_id(mapa[k])] = eje_mayor(g)
    return res
 
 
def unidad_mayoritaria(buffers, capa, campo):
    tmp = "memory/f5_inter"
    arcpy.analysis.PairwiseIntersect([buffers, capa], tmp)
    t = pd.DataFrame([r for r in arcpy.da.SearchCursor(tmp, ["EV_ID", campo, "SHAPE@AREA"])],
                     columns=["EV_ID", "UNI", "AREA"])
    arcpy.management.Delete(tmp)
    t = t.dropna(subset=["UNI"])
    t["UNI"] = t["UNI"].astype(str).str.strip()
    a = t.groupby(["EV_ID", "UNI"], as_index=False)["AREA"].sum()
    tot = a.groupby("EV_ID")["AREA"].transform("sum")
    a["PUREZA"] = a["AREA"] / tot
    a = a.sort_values("AREA", ascending=False).drop_duplicates("EV_ID")
    return a.set_index("EV_ID")[["UNI", "PUREZA"]].to_dict("index")
 

# Preparación, exportación y diagnóstico 

### ¿Que hace?

Arma la capa de eventos con todos los campos nuevos, la exporta e imprime el diagnóstico.

Se divide en ocho secciones:

1. Comprueba que existan los CSV y las capas con su tipo de geometría y su campo, y crea la geodatabase y la carpeta de salida. Toma el sistema de coordenadas de la capa de fallas.

2. Une la tabla final de la Fase 3 con la del modelo (sin repetir columnas) y con la región de SKATER, y avisa si algún evento no estaba en el modelo.

3. Traza el círculo de 18 m alrededor de cada corona y, para cada capa de unidades, guarda la unidad mayoritaria

4. Guarda la orientación y la elongación del rectangulo de cada evento

5. busca la falla más cercana a cada corona y el punto de la falla donde se da esa distancia.  La orientación de la falla se mide en un pedazo de 200 m centrado en ese punto, así que en una falla curva cuenta el tramo cercano y no la línea completa. 

6.  Corta cada falla en tramos de 200 m (el último se conserva si mide al menos 100 m), calcula la orientación de cada uno y los recorta al área de estudio.

7. EXxporta las tablas

8. Genera el diagnostico, imprime en consola:

 * eventos y anomalías por clase de los factores agrupados y del tipo
 * las unidades sin agrupar con su pureza mediana, si pasan los cortes de 15 y 20 eventos, los eventos sin unidad y las unidades repartidas en más de un grupo
 * anomalías por tercil de pendiente, densidad de lineamientos y distancia al drenaje
 * la distancia a la falla
 * la alineación: eventos sin orientación, control de la elongación con la de la Fase 1, y mediana del ángulo y porcentaje de ángulos de hasta 30° por grupo y filtro
 * las anomalías por región de SKATER y el número y largo de los tramos de falla.

In [39]:

for ruta in (CSV_EVENTOS_R, TABLA_MODELO):
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No existe: {ruta}")
verificar_capa(FALLAS, tipo_geom="Polyline")
for nuevo, (capa, campo) in UNIDADES.items():
    verificar_capa(capa, campo, tipo_geom="Polygon")
 
os.makedirs(os.path.dirname(GDB_OUT), exist_ok=True)
if not arcpy.Exists(GDB_OUT):
    arcpy.management.CreateFileGDB(os.path.dirname(GDB_OUT), os.path.basename(GDB_OUT))
os.makedirs(CSV_DIR, exist_ok=True)
sr = arcpy.Describe(FALLAS).spatialReference
 
# =====================================================================
# 1. TABLA DE EVENTOS (pandas) Y CAPA DE PUNTOS
# =====================================================================
ev = leer_csv(CSV_EVENTOS_R)
mod = leer_csv(TABLA_MODELO)
mod = mod.drop(columns=[c for c in mod.columns if c in ev.columns and c != F_ID])
ev = ev.merge(mod, on=F_ID, how="left")
if "EN_MODELO" in ev.columns:
    fuera = int((ev["EN_MODELO"] != 1).sum())
    if fuera:
        print(f"  ! {fuera} eventos del CSV integrado no están marcados EN_MODELO")
 
hay_region = bool(TABLA_REGIONES) and os.path.exists(TABLA_REGIONES)
if hay_region:
    reg = leer_csv(TABLA_REGIONES)
    if F_REG not in reg.columns:
        raise ValueError(f"'{F_REG}' no está en la tabla de regiones: {list(reg.columns)}")
    ev = ev.merge(reg[[F_ID, F_REG]].drop_duplicates(F_ID), on=F_ID, how="left")
else:
    print("  ! Sin tabla de regiones de la Fase 4: se omite la región")
 
rd2 = ev[F_DR] ** 2
ev["ANOM"] = (rd2 > UMB_MOD).astype(int)
ev["ANOM_CLASE"] = np.where(rd2 > UMB_FUE, "Fuerte", np.where(rd2 > UMB_MOD, "Moderada", "Normal"))
if "ANOMALIA" in ev.columns:
    distintos = int((ev["ANOM"] != (ev["ANOMALIA"].str.lower() != "normal").astype(int)).sum())
    if distintos:
        print(f"  ! {distintos} eventos con etiqueta distinta a la columna ANOMALIA de la Fase 3")
 
csv_entrada = os.path.join(CSV_DIR, "F5_eventos_entrada.csv")
ev.to_csv(csv_entrada, index=False, encoding="utf-8")
EV = os.path.join(GDB_OUT, "F5_eventos")
arcpy.management.XYTableToPoint(csv_entrada, EV, F_X, F_Y, coordinate_system=sr)
n_ev = int(arcpy.management.GetCount(EV)[0])
 
# =====================================================================
# 2. UNIDADES SIN AGRUPAR (mayoritaria en 22 m)
# =====================================================================
buf = "memory/f5_buf"
arcpy.analysis.PairwiseBuffer(EV, buf, f"{RADIO_BUFFER_M} Meters")
agregar_campo(buf, "EV_ID", "DOUBLE")
arcpy.management.CalculateField(buf, "EV_ID", f"!{F_ID}!", "PYTHON3")
sobran = [f.name for f in arcpy.ListFields(buf) if not f.required and f.name != "EV_ID"]
if sobran:
    arcpy.management.DeleteField(buf, sobran)
 
for nuevo, (capa, campo) in UNIDADES.items():
    res = unidad_mayoritaria(buf, capa, campo)
    agregar_campo(EV, nuevo, "TEXT")
    agregar_campo(EV, f"PUR_{nuevo}", "DOUBLE")
    with arcpy.da.UpdateCursor(EV, [F_ID, nuevo, f"PUR_{nuevo}"]) as cur:
        for idv, _, _ in cur:
            r = res.get(float(idv))
            cur.updateRow([idv, r["UNI"] if r else None, r["PUREZA"] if r else None])
arcpy.management.Delete(buf)
 
# =====================================================================
# 3. ORIENTACIÓN DEL EJE MAYOR DE CADA EVENTO
# =====================================================================
for nombre, tipo in [("ORIENT_AX", "DOUBLE"), ("ELONG_RECT", "DOUBLE"), ("ORIENT_OK", "SHORT")]:
    agregar_campo(EV, nombre, tipo)
rect = leer_rectangulos()
with arcpy.da.UpdateCursor(EV, [F_ID, "ORIENT_AX", "ELONG_RECT", "ORIENT_OK"]) as cur:
    for idv, *_ in cur:
        o, e = rect.get(clave_id(idv), (None, None))
        cur.updateRow([idv, o, e, 1 if (e is not None and e >= E_MIN) else 0])
 
# =====================================================================
# 4. DISTANCIA Y ALINEACIÓN CON LA FALLA MÁS CERCANA
# =====================================================================
arcpy.analysis.Near(EV, FALLAS, location="LOCATION", method="PLANAR")
oid_f = arcpy.Describe(FALLAS).OIDFieldName
geoms = {o: g for o, g in arcpy.da.SearchCursor(FALLAS, [oid_f, "SHAPE@"], spatial_reference=sr)}
 
for nombre in ("D_FALLA", "ORIENT_FALLA", "ANG_REL", "ANG_REL_S"):
    agregar_campo(EV, nombre, "DOUBLE")
campos = ["NEAR_FID", "NEAR_DIST", "NEAR_X", "NEAR_Y", "ORIENT_AX",
          "D_FALLA", "ORIENT_FALLA", "ANG_REL", "ANG_REL_S"]
with arcpy.da.UpdateCursor(EV, campos) as cur:
    for nfid, ndist, nx, ny, o_ev, *_ in cur:
        g = geoms.get(nfid)
        o_f = ang = ang_s = None
        if g is not None and g.length > 0:
            m = g.measureOnLine(arcpy.PointGeometry(arcpy.Point(nx, ny), sr))
            a, b = max(0.0, m - TRAMO_M / 2), min(g.length, m + TRAMO_M / 2)
            s = g.segmentAlongLine(a, b) if b - a > 1 else g
            o_f = orientacion(s.firstPoint, s.lastPoint)
            if o_ev is not None:
                ang_s = ((o_ev - o_f + 90.0) % 180.0) - 90.0    # [-90, 90): diferencia axial con signo
                ang = abs(ang_s)                                  # 0 = paralelo, 90 = perpendicular
        cur.updateRow([nfid, ndist, nx, ny, o_ev, ndist, o_f, ang, ang_s])
arcpy.management.DeleteField(EV, ["NEAR_FID", "NEAR_DIST", "NEAR_X", "NEAR_Y"])
 
# =====================================================================
# 5. TRAMOS DE FALLA CON ORIENTACIÓN AXIAL (contexto global)
# =====================================================================
seg_tmp = os.path.join(GDB_OUT, "F5_estructuras_seg_tmp")
arcpy.management.CreateFeatureclass(GDB_OUT, "F5_estructuras_seg_tmp", "POLYLINE", spatial_reference=sr)
for nombre, tipo in [("FUENTE", "TEXT"), ("ORIENT_AX", "DOUBLE"), ("LONG_M", "DOUBLE")]:
    agregar_campo(seg_tmp, nombre, tipo)
with arcpy.da.InsertCursor(seg_tmp, ["SHAPE@", "FUENTE", "ORIENT_AX", "LONG_M"]) as ins:
    for (g,) in arcpy.da.SearchCursor(FALLAS, ["SHAPE@"], spatial_reference=sr):
        if g is None or g.length == 0:
            continue
        ini = 0.0
        while ini < g.length:
            fin = min(ini + TRAMO_M, g.length)
            if fin - ini >= TRAMO_M / 2:
                s = g.segmentAlongLine(ini, fin)
                ins.insertRow([s, "FALLA_MZ", orientacion(s.firstPoint, s.lastPoint), fin - ini])
            ini = fin
SEG = os.path.join(GDB_OUT, "F5_estructuras_seg")
if AREA_ESTUDIO and arcpy.Exists(AREA_ESTUDIO):
    arcpy.analysis.Clip(seg_tmp, AREA_ESTUDIO, SEG)
    arcpy.management.Delete(seg_tmp)
else:
    arcpy.management.Rename(seg_tmp, SEG)
 
# =====================================================================
# 6. EXPORTACIÓN PARA R
# =====================================================================
df = exportar_csv(EV, os.path.join(CSV_DIR, "F5_eventos.csv"))
seg = exportar_csv(SEG, os.path.join(CSV_DIR, "F5_estructuras_seg.csv"))
 
# =====================================================================
# 7. DIAGNÓSTICO
# =====================================================================
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)
an = df["ANOM"] == 1
print("=" * 78)
print(f"Eventos: {n_ev} | anomalías: {int(an.sum())} "
      f"(fuertes {int((df['ANOM_CLASE'] == 'Fuerte').sum())}) | tasa global {100 * an.mean():.1f} %")
 
 
def resumen(campo):
    t = df.groupby(campo, dropna=False).agg(N_EV=("ANOM", "size"), N_ANOM=("ANOM", "sum"))
    t["%_ANOM"] = (100 * t["N_ANOM"] / t["N_EV"]).round(1)
    return t.sort_values("N_EV", ascending=False)
 
 
print("\n--- Factores agrupados (los de la GWR) y tipo ---")
for campo in ["GEOLOGIA", "GEOMORFOLOGIA", "COBERTURA", "TIPO"]:
    print(f"\n{campo}\n{resumen(campo).to_string()}")
 
print("\n--- Unidades sin agrupar (regla del asesor) ---")
for nuevo in UNIDADES:
    t = resumen(nuevo)
    t["PUREZA_MED"] = df.groupby(nuevo)[f"PUR_{nuevo}"].median().round(2)
    t["ENTRA"] = np.where(t["N_EV"] >= MIN_EV_UNIDAD[1], f">={MIN_EV_UNIDAD[1]}",
                          np.where(t["N_EV"] >= MIN_EV_UNIDAD[0], f">={MIN_EV_UNIDAD[0]}", "no"))
    print(f"\n{nuevo}\n{t.to_string()}")
    sin = int(df[nuevo].isna().sum())
    for corte in MIN_EV_UNIDAD:
        ok = t[t["N_EV"] >= corte]
        print(f"  corte {corte}: {len(ok)} de {len(t)} unidades | "
              f"{int(ok['N_EV'].sum())} eventos y {int(ok['N_ANOM'].sum())} anomalías conservados")
    if sin:
        print(f"  ! {sin} eventos sin unidad (círculo fuera de la capa)")
    mezcla = df.dropna(subset=[nuevo]).groupby(nuevo)[AGRUPADO[nuevo]].nunique()
    mezcla = mezcla[mezcla > 1]
    if len(mezcla):
        print(f"  ! Unidades repartidas en más de un grupo de {AGRUPADO[nuevo]}: {list(mezcla.index)}")
 
print("\n--- Terciles de las continuas de la GWR (estructura residual) ---")
for campo in ["PENDIENTE", "DENS_LIN_1000", "DIST_DRENAJE"]:
    if campo in df.columns:
        t = df.groupby(pd.qcut(df[campo], 3), observed=True).agg(
            N_EV=("ANOM", "size"), N_ANOM=("ANOM", "sum"))
        print(f"\n{campo}\n{t.to_string()}")
 
print("\n--- Distancia a la falla ---")
print(f"Mediana: anómalos {df.loc[an, 'D_FALLA'].median():.0f} m | resto {df.loc[~an, 'D_FALLA'].median():.0f} m")
if "DIST_ESTRUCTURA" in df.columns:
    dif = (df["D_FALLA"] - df["DIST_ESTRUCTURA"]).abs()
    print(f"Control con DIST_ESTRUCTURA de la Fase 3: diferencia máxima {dif.max():.1f} m")
bordes = list(range(0, DMAX_M + BANDA_M, BANDA_M)) + [np.inf]
t = df.groupby(pd.cut(df["D_FALLA"], bordes, right=False), observed=False).agg(
    N_EV=("ANOM", "size"), N_ANOM=("ANOM", "sum"))
t["PROP_ANOM"] = (t["N_ANOM"] / t["N_EV"]).round(3)
print(t.to_string())
 
print("\n--- Alineación con la falla más cercana ---")
print(f"Eventos sin rectángulo: {int(df['ORIENT_AX'].isna().sum())} | "
      f"sin orientación de falla: {int(df['ORIENT_FALLA'].isna().sum())}")
if "LOG_ELONG" in df.columns:
    rel = (df["ELONG_RECT"] / 10 ** df["LOG_ELONG"] - 1).abs()
    print(f"Control con la elongación de la Fase 1: diferencia relativa mediana {100 * rel.median():.1f} % "
          f"| eventos con más de 5 %: {int((rel > 0.05).sum())}")
cerca = df["D_FALLA"] <= DIST_ALINEACION_M
ok = df["ORIENT_OK"] == 1
filas = []
for nombre, sel in [("Anómalos", an), ("Resto", ~an)]:
    for etiqueta, filtro in [(f"E>={E_MIN}", ok), (f"E>={E_MIN} y <= {DIST_ALINEACION_M} m", ok & cerca)]:
        x = df.loc[sel & filtro, "ANG_REL"]
        filas.append([nombre, etiqueta, len(x), round(x.median(), 1), round(100 * (x <= 30).mean(), 1)])
print(pd.DataFrame(filas, columns=["Grupo", "Filtro", "N", "Mediana_ang", "%_<=30°"]).to_string(index=False))
 
if hay_region:
    print(f"\n--- Región de la Fase 4 ({F_REG}) ---")
    print(f"Eventos sin región: {int(df[F_REG].isna().sum())}")
    print(resumen(F_REG).to_string())
 
print("\nTramos de falla:", len(seg), f"| {seg['LONG_M'].sum() / 1000:.1f} km")
print("=" * 78)

Rectángulos ajustados a Inventario_integrado (campo de ID: ID_evento)
Eventos: 427 | anomalías: 19 (fuertes 13) | tasa global 4.4 %

--- Factores agrupados (los de la GWR) y tipo ---

GEOLOGIA
                                N_EV  N_ANOM  %_ANOM
GEOLOGIA                                            
Rocas Ígneas                     194       9     4.6
Rocas Metamórficas               108       4     3.7
Depósitos Neógeno-Cuaternarios    63       3     4.8
Rocas Volcano-Sedimentarias       62       3     4.8

GEOMORFOLOGIA
                                                         N_EV  N_ANOM  %_ANOM
GEOMORFOLOGIA                                                                
Relieves denudacionales residuales (colinas y lomos)      326      16     4.9
Relieves depositacionales y superficies de aplanamiento    60       2     3.3
Relieves estructurales, tectónicos y antrópicos            41       1     2.4

COBERTURA
                                N_EV  N_ANOM  %_ANOM
COBERTURA           